# Run `feat/inference-app` integration tests on a remote GPU runtime

This notebook is intended for Colab or another hosted Jupyter environment.

It will:
- clone `https://github.com/phuongthao0515/Magma.git`
- check out `feat/inference-app`
- install the Python and system dependencies needed by the real `app/tests/integration` path
- use an existing local adapter checkpoint if present, otherwise download it from Google Drive
- switch into the `app/` folder and point the test run at the integration scenarios manifest
- run `python -m pytest tests/integration -m integration -rs -vv`
- show the latest saved SoM image and structured model output from the most recent test step

Notes:
- Use a GPU runtime if possible. `Magma-8B` on CPU will be very slow and may run out of memory.
- The first real test run will also download `microsoft/Magma-8B` and `microsoft/OmniParser-v2.0` if they are not cached.
- The adapter checkpoint download tries the new Drive folder first, then the old folder as a fallback.
- Colab clones the remote GitHub branch state, not any uncommitted local files from your IDE.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

REPO_URL = "https://github.com/phuongthao0515/Magma.git"
REPO_BRANCH = "feat/inference-app"
WORKSPACE_ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
REPO_DIR = WORKSPACE_ROOT / "Magma"
APP_DIR = REPO_DIR / "app"

if not REPO_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            REPO_BRANCH,
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )
else:
    print(f"Reusing existing clone at {REPO_DIR}")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH, "--depth", "1"], check=True)
    checkout = subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
    if checkout.returncode != 0:
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-b", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

os.chdir(REPO_DIR)
current_branch = subprocess.run(
    ["git", "branch", "--show-current"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print(f"Repo dir: {REPO_DIR}")
print(f"App dir: {APP_DIR}")
print(f"Branch: {current_branch}")
print(f"Python: {sys.version.split()[0]}")


In [ ]:
import importlib.util
import shutil


def pip_install(*packages: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"],
    check=True,
)

if shutil.which("apt-get"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "ffmpeg", "libgl1", "libglib2.0-0"],
        check=True,
    )
else:
    print("apt-get not available; skipping OS package install")

required_modules = {
    "pytest": "pytest",
    "pytest_asyncio": "pytest-asyncio",
    "fastapi": "fastapi",
    "uvicorn": "uvicorn",
    "multipart": "python-multipart",
    "pydantic": "pydantic",
    "httpx": "httpx",
    "dotenv": "python-dotenv",
    "numpy": "numpy",
    "PIL": "pillow",
    "gdown": "gdown",
    "huggingface_hub": "huggingface-hub",
    "peft": "peft",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "cv2": "opencv-python-headless",
    "ultralytics": "ultralytics",
    "easyocr": "easyocr",
    "open_clip": "open_clip_torch",
    "sentencepiece": "sentencepiece",
    "einops": "einops",
    "einops_exts": "einops-exts",
    "timm": "timm",
    "torch": "torch",
    "torchvision": "torchvision",
}

missing_packages = [
    package_name
    for module_name, package_name in required_modules.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print(f"Installing missing packages: {missing_packages}")
    pip_install(*missing_packages)
else:
    print("Required Python packages already available")

print("Installing Magma-compatible transformers fork...")
pip_install("git+https://github.com/jwyang/transformers.git@dev/jwyang-v4.48.2")


In [ ]:
import json
import shutil
import time

import gdown
import torch

PRIMARY_CHECKPOINT_URL = "https://drive.google.com/drive/folders/15Q9pnO5pTq22qDcZi-hYRSY4M34nb20U?usp=sharing"
FALLBACK_CHECKPOINT_URL = "https://drive.google.com/drive/folders/18RNkzvGCehTvi6J1vqV4hb8E9_fkmvXR?usp=drive_link"
CHECKPOINT_URLS = [PRIMARY_CHECKPOINT_URL, FALLBACK_CHECKPOINT_URL]

CHECKPOINT_DIR = APP_DIR / "checkpoints" / "finetune-3apps-r32-a64-maxlen2560-focal-marks" / "checkpoint-3400"
WEIGHTS_DIR = APP_DIR / "server" / "weights"
SCENARIOS_PATH = APP_DIR / "tests" / "fixtures" / "integration" / "scenarios.json"
SCENARIOS_TEMPLATE_PATH = APP_DIR / "tests" / "fixtures" / "integration" / "scenarios.example.json"
PYTEST_BASETEMP = APP_DIR / ".pytest_tmp"


def has_valid_checkpoint(path: Path) -> bool:
    return (path / "adapter_config.json").exists() and (
        (path / "adapter_model.safetensors").exists() or (path / "adapter_model.bin").exists()
    )


def download_checkpoint_if_missing(output_dir: Path, urls: list[str], retries: int = 3) -> None:
    if has_valid_checkpoint(output_dir):
        print(f"Checkpoint already present at {output_dir}")
        return

    last_exc = None
    for index, url in enumerate(urls, start=1):
        print(f"Trying checkpoint source {index}/{len(urls)}: {url}")

        if output_dir.exists():
            shutil.rmtree(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        for attempt in range(1, retries + 1):
            try:
                print(f"  Download attempt {attempt}/{retries} with use_cookies=False")
                gdown.download_folder(
                    url=url,
                    output=str(output_dir),
                    quiet=False,
                    use_cookies=False,
                    remaining_ok=False,
                )
                if has_valid_checkpoint(output_dir):
                    print(f"Checkpoint downloaded successfully to {output_dir}")
                    return
                raise RuntimeError("Checkpoint download finished, but adapter files are missing")
            except Exception as exc:
                last_exc = exc
                print(f"  Attempt failed: {exc}")
                if output_dir.exists() and not has_valid_checkpoint(output_dir):
                    shutil.rmtree(output_dir)
                    output_dir.mkdir(parents=True, exist_ok=True)
                if attempt < retries:
                    time.sleep(5 * attempt)

        try:
            print("  Retrying with use_cookies=True")
            gdown.download_folder(
                url=url,
                output=str(output_dir),
                quiet=False,
                use_cookies=True,
                remaining_ok=False,
            )
            if has_valid_checkpoint(output_dir):
                print(f"Checkpoint downloaded successfully to {output_dir}")
                return
            raise RuntimeError("Checkpoint download finished, but adapter files are missing")
        except Exception as exc:
            last_exc = exc
            print(f"  Source failed: {exc}")
            if output_dir.exists() and not has_valid_checkpoint(output_dir):
                shutil.rmtree(output_dir)

    raise RuntimeError(
        f"Checkpoint missing at {output_dir} after trying all URLs: {urls}"
    ) from last_exc


os.environ.setdefault("HF_HOME", str(WORKSPACE_ROOT / ".cache" / "huggingface"))
os.environ["MAGMA_CHECKPOINT_DIR"] = str(CHECKPOINT_DIR)
os.environ["MAGMA_CHECKPOINT_URLS"] = ",".join(CHECKPOINT_URLS)
os.environ["OMNIPARSER_WEIGHTS_DIR"] = str(WEIGHTS_DIR)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

download_checkpoint_if_missing(CHECKPOINT_DIR, CHECKPOINT_URLS)

if not SCENARIOS_PATH.exists():
    raise FileNotFoundError(
        f"Missing integration manifest: {SCENARIOS_PATH}\n"
        "Colab cloned the remote GitHub branch, not your local IDE workspace.\n"
        "If `scenarios.json` exists only locally, commit/push it to `feat/inference-app` or create/upload it in Colab.\n"
        f"Template file: {SCENARIOS_TEMPLATE_PATH}"
    )

os.environ["FULL_APP_INTEGRATION_SCENARIOS"] = str(SCENARIOS_PATH)

os.chdir(APP_DIR)
manifest = json.loads(SCENARIOS_PATH.read_text(encoding="utf-8"))

print(f"Working directory: {Path.cwd()}")
print(f"Scenarios file: {SCENARIOS_PATH}")
print(f"Scenario names: {', '.join(sorted(manifest.keys()))}")
print(f"HF_HOME: {os.environ['HF_HOME']}")
print(f"MAGMA_CHECKPOINT_DIR: {CHECKPOINT_DIR}")
print(f"MAGMA_CHECKPOINT_URLS: {CHECKPOINT_URLS}")
print(f"OMNIPARSER_WEIGHTS_DIR: {WEIGHTS_DIR}")
print(f"PYTEST_BASETEMP: {PYTEST_BASETEMP}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: GPU runtime not detected. The real Magma integration tests may be too slow on CPU.")


In [ ]:
env = os.environ.copy()
env["FULL_APP_USE_REAL_MODEL_TESTS"] = "1"
env["PYTHONPATH"] = str(APP_DIR / "server") + os.pathsep + env.get("PYTHONPATH", "")

if PYTEST_BASETEMP.exists():
    shutil.rmtree(PYTEST_BASETEMP)

pytest_run = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/integration",
        "-m",
        "integration",
        "-rs",
        "-vv",
        "--basetemp",
        str(PYTEST_BASETEMP),
    ],
    env=env,
)

PYTEST_RETURN_CODE = pytest_run.returncode
print(f"Pytest exit code: {PYTEST_RETURN_CODE}")
print(f"Pytest artifacts directory: {PYTEST_BASETEMP}")

if PYTEST_RETURN_CODE != 0:
    print()
    print("Pytest did not complete successfully.")
    print("The real failure is in the pytest output above this message.")
    print("This cell intentionally does not raise CalledProcessError, so you can still inspect saved artifacts below.")


In [ ]:
from IPython.display import display
from PIL import Image

result_files = sorted(
    PYTEST_BASETEMP.rglob("step_*_model_output.json"),
    key=lambda path: path.stat().st_mtime_ns,
)

if not result_files:
    raise FileNotFoundError(
        f"No saved model output found under {PYTEST_BASETEMP}. "
        "Run the pytest cell first. If pytest already ran and the exit code was non-zero, "
        "the failure likely happened before the first inference step saved artifacts."
    )

latest_result_path = result_files[-1]
latest_payload = json.loads(latest_result_path.read_text(encoding="utf-8"))
som_image_path = Path(latest_payload["som_image_path"]) if latest_payload.get("som_image_path") else None

print(f"Latest model output file: {latest_result_path}")
print(f"Task ID: {latest_payload['task_id']}")
print(f"Step: {latest_payload['step']}")
print("Prompt:")
print(latest_payload["prompt"])
print()
print("Previous actions:")
print(latest_payload["previous_actions"])
print()
print("Model output:")
print(json.dumps(latest_payload["model_output"], indent=2))

if som_image_path is not None and som_image_path.exists():
    print()
    print(f"Latest SoM image: {som_image_path}")
    display(Image.open(som_image_path))
else:
    print()
    print("No SoM image was saved for the latest step.")


In [ ]:
from collections import defaultdict

result_files = sorted(
    PYTEST_BASETEMP.rglob("step_*_model_output.json"),
    key=lambda path: path.stat().st_mtime_ns,
)

if not result_files:
    raise FileNotFoundError(
        f"No saved model output found under {PYTEST_BASETEMP}. Run the pytest cell first."
    )

latest_per_test = {}
for result_path in result_files:
    test_dir = next((parent for parent in result_path.parents if parent.name.startswith("test_")), None)
    test_name = test_dir.name if test_dir is not None else "unknown_test"
    latest_per_test[test_name] = result_path

print(f"Found {len(latest_per_test)} test result groups under {PYTEST_BASETEMP}")

for test_name, result_path in sorted(latest_per_test.items()):
    payload = json.loads(result_path.read_text(encoding="utf-8"))
    som_image_path = Path(payload["som_image_path"]) if payload.get("som_image_path") else None

    print("=" * 80)
    print(f"Test directory: {test_name}")
    print(f"Model output file: {result_path}")
    print(f"Task ID: {payload['task_id']}")
    print(f"Step: {payload['step']}")
    print("Prompt:")
    print(payload["prompt"])
    print()
    print("Previous actions:")
    print(payload["previous_actions"])
    print()
    print("Model output:")
    print(json.dumps(payload["model_output"], indent=2))

    if som_image_path is not None and som_image_path.exists():
        print()
        print(f"SoM image: {som_image_path}")
        display(Image.open(som_image_path))
    else:
        print()
        print("No SoM image was saved for this test result.")
